# Machine Learning Model Development

The objective of this notebook is to develop regression models capable of predicting particulate matter concentrations (PM10 and PM2.5) using meteorological parameters. Multiple machine learning algorithms are trained and evaluated to identify the model that best captures the relationship between weather conditions and dust concentration within the mining environment.

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor,
    VotingRegressor
)
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
df = pd.read_excel("../data/processed/mine_data_cleaned.xlsx")
df.columns = df.columns.str.strip()
df.head()

,YEAR,SEASON,MONTH,LOCATION,WEEK,Temperature,Relative_humidity,RF-CUMULATIVE,Wind_speed,PM10,PM2.5
0,2023,WINTER,DECEMBER,HAUL ROAD,1,30.1,64.5,1.0,2.55,56.8,42.6
1,2023,WINTER,DECEMBER,HAUL ROAD,1,29.2,70.3,1.0,2.55,62.4,41.9
2,2023,WINTER,DECEMBER,HAUL ROAD,2,27.6,68.0,0.0,2.55,63.9,43.6
3,2023,WINTER,DECEMBER,HAUL ROAD,2,24.5,85.3,0.0,2.55,65.7,41.9
4,2024,WINTER,DECEMBER,HAUL ROAD,3,28.8,58.9,0.0,2.55,60.2,39.5


In [4]:
features = [
    "Temperature",
    "Relative_humidity",
    "RF-CUMULATIVE",
    "Wind_speed"
]

target_pm10 = "PM10"
target_pm25 = "PM2.5"

In [5]:
X = df[features]
y = df[target_pm10]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [6]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred = lr.predict(X_test)

In [8]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(mae)
print(rmse)
print(r2)

9.298022279832548
11.508841943569998
0.10765327237297229


In [9]:
# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [10]:
mae = mean_absolute_error(y_test, dt_pred)
rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
r2 = r2_score(y_test, dt_pred)

print(mae)
print(rmse)
print(r2)

6.943455565572389
9.309794225482335
0.4160837686988156


In [11]:
# Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [12]:
mae = mean_absolute_error(y_test, rf_pred)
rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
r2 = r2_score(y_test, rf_pred)

print(mae)
print(rmse)
print(r2)

6.922561912446176
9.245581441368868
0.4241109234497611


In [13]:
# Bagging
try:
    bagging = BaggingRegressor(
        estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )
except TypeError:
    bagging = BaggingRegressor(
        base_estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )

bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)

In [14]:
mae = mean_absolute_error(y_test, bagging_pred)
rmse = np.sqrt(mean_squared_error(y_test, bagging_pred))
r2 = r2_score(y_test, bagging_pred)

print(mae)
print(rmse)
print(r2)

6.920779695090471
9.248180649265292
0.42378707884355293


In [15]:
# SVR
svr = SVR(kernel="rbf")
svr.fit(X_train, y_train)
svr_pred = svr.predict(X_test)

In [16]:
mae = mean_absolute_error(y_test, svr_pred)
rmse = np.sqrt(mean_squared_error(y_test, svr_pred))
r2 = r2_score(y_test, svr_pred)

print(mae)
print(rmse)
print(r2)

9.002595075384619
11.419584802919418
0.12144083856015742


In [17]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

In [18]:
mae = mean_absolute_error(y_test, xgb_pred)
rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
r2 = r2_score(y_test, xgb_pred)

print(mae)
print(rmse)
print(r2)

7.008850817971835
9.082760347121875
0.44421593042108565


In [19]:
# Compare model performance in a table

results = []

models = {
    "Linear Regression": pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "Bagging": bagging_pred,
    "SVR": svr_pred,
    "XGBoost": xgb_pred,
}

for model_name, y_pred in models.items():
    results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R²": r2_score(y_test, y_pred)
    })

# Voting Regressor
try:
    voting = VotingRegressor([
        ("lr", LinearRegression()),
        ("dt", DecisionTreeRegressor(random_state=42)),
        ("rf", RandomForestRegressor(n_estimators=200, random_state=42)),
        ("xgb", xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=200,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ))
    ])
    voting.fit(X_train, y_train)
    voting_pred = voting.predict(X_test)

    results.append({
        "Model": "Voting",
        "MAE": mean_absolute_error(y_test, voting_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, voting_pred)),
        "R²": r2_score(y_test, voting_pred)
    })
except Exception as e:
    print("Voting model could not be evaluated:", e)

results_df = pd.DataFrame(results)
results_df

,Model,MAE,RMSE,R²
0,Linear Regression,9.298022,11.508842,0.107653
1,Decision Tree,6.943456,9.309794,0.416084
2,Random Forest,6.922562,9.245581,0.424111
3,Bagging,6.920780,9.248181,0.423787
4,SVR,9.002595,11.419585,0.121441
5,XGBoost,7.008851,9.082760,0.444216
6,Voting,7.016261,9.170590,0.433415


In [21]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
results_df.to_excel(
    "../data/processed/results_data.xlsx",
    index=False
)